# Industrial Anomaly Detection

A normal-only bottle inspection baseline with frozen ResNet18 features and spatial diagonal Gaussian statistics. This notebook reads the committed results and fitted model. Optional cells reproduce the benchmark after downloading the data.

**Question:** can a model fitted only on good bottles detect broken or contaminated bottles?

In [1]:
import json
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
REPORT = ROOT / "reports/bottle"
metrics = json.loads((REPORT / "metrics.json").read_text())
print("Project:", ROOT.name)

Project: industrial-anomaly-detection


## Keep the splits separate

167 normal images fit the statistics; 42 separate normal images select the threshold. All 83 official test images remain untouched until evaluation. Split manifests include content hashes.

In [2]:
calibration = json.loads((REPORT / "calibration.json").read_text())
manifest = json.loads((REPORT / "training_manifest.json").read_text())
fit_hashes = {row["sha256"] for row in manifest if row["split"] == "fit"}
calibration_hashes = {row["sha256"] for row in manifest if row["split"] == "calibration"}
assert not fit_hashes & calibration_hashes
print("Fitting normals:", len(fit_hashes))
print("Calibration normals:", len(calibration_hashes))
print("Threshold:", calibration["threshold"])

Fitting normals: 167
Calibration normals: 42
Threshold: 3.8109042644500732


## Read the fitted normal model

For each of the 192 channels and 32 × 32 positions, keep a mean and regularized variance. The encoder is frozen; no defect labels or neural optimizer are involved.

In [3]:
with np.load(ROOT / "models/bottle_gaussian.npz", allow_pickle=False) as model:
    print("Mean shape:", model["mean"].shape)
    print("Variance shape:", model["variance"].shape)
    assert np.all(model["variance"] > 0)

Mean shape: (192, 32, 32)
Variance shape: (192, 32, 32)


## Evaluate ranking and actual flags

Image AUROC measures score ranking. The fixed normal-calibration threshold determines false alarms and detected defects. Pixel AUROC uses resized 256 × 256 reference masks; it is not AU-PRO.

In [4]:
for key in [
    "test_images",
    "image_auroc",
    "pixel_auroc_at_256",
    "anomaly_recall",
    "normal_false_positive_rate",
]:
    print(f"{key}: {metrics[key]}")
print("Confusion matrix [normal, defective]:", metrics["confusion_matrix"])

test_images: 83
image_auroc: 1.0
pixel_auroc_at_256: 0.9839883788776189
anomaly_recall: 1.0
normal_false_positive_rate: 0.1
Confusion matrix [normal, defective]: [[18, 2], [0, 63]]


![Evaluation curves](../reports/bottle/evaluation.png)

## Inspect false alarms and defect locations

The figure shows two false alarms, one near-threshold example per defect category, and a nearby remaining decision. Feature distance overlays are not probabilities or causal explanations.

![Visual inspection](../reports/bottle/inspection.png)

Image attribution: © MVTec Software GmbH; adapted figure under CC BY-NC-SA 4.0. See [NOTICE](../NOTICE.md).

In [5]:
examples = json.loads((REPORT / "inspection_examples.json").read_text())
for example in examples:
    print(example["path"], "correct" if example["correct"] else "false alarm / missed defect")

test/good/006.png false alarm / missed defect
test/good/015.png false alarm / missed defect
test/broken_large/011.png correct
test/broken_small/007.png correct
test/contamination/019.png correct
test/good/019.png correct


## Reproduce fitting and evaluation

Install the package with `pip install -e .`, run `python scripts/download_data.py` from the root, then set `REPRODUCE=True`. ResNet18 weights download once. The archive mirror is checksum-pinned. This optional cell is disabled in the committed notebook to avoid refitting during casual review.

In [6]:
REPRODUCE = False
if REPRODUCE:
    import torch

    from inspection.cli import evaluate, fit

    torch.set_num_threads(4)
    fit(ROOT / "data/bottle", ROOT / "models/bottle_gaussian.npz", REPORT)
    evaluate(ROOT / "data/bottle", ROOT / "models/bottle_gaussian.npz", REPORT)
else:
    print("Using committed results. Set REPRODUCE=True after downloading the data.")

Using committed results. Set REPRODUCE=True after downloading the data.


## Try another image

From the root:
```bash
inspect-bottles predict --image data/bottle/test/broken_small/000.png --output reports/prediction.png
```

## What remains uncertain?

This single-category run detected 63/63 defects and incorrectly flagged 2/20 normals. New lighting, alignment, bottle types, or categories can change performance. A small calibration set does not establish a production false-alarm rate. Next experiments should use separate validation data for model selection and evaluate shifts across several categories.

[Methodology](../docs/methodology.md) · [Dataset](https://www.mvtec.com/research-teaching/datasets/mvtec-ad) · [PaDiM context](https://arxiv.org/abs/2011.08785)